# M-0002 — Well-code demultiplex（按 384 well UDI 拆分 FASTQ）（v1）

状态：**Complete**  
最后更新：2026-01-15

## 1) Context / Goal

- M-0001 已完成：Locked `demultiplex (jfjlaros)` as the core tool.
- **Goal**: Implement the first stage of demultiplexing: splitting raw reads by **Well Code (384 UDI)**.
- Output: Individual FASTQ files per well, plus a counts summary table. Unknown/unmatched reads must be preserved.

## 2) Scope / Requirements

- **Tool**: Wrap `demultiplex` (jfjlaros) in a python script (`scripts/run_demux.py`).
- **Input**: R1 FASTQ (optional R2), Barcode CSV.
- **Output**: `out/well/<ID>.fq.gz`, `out/unknown.fq.gz`, `stats.tsv`.
- **Constraints**:
  - **Environment**: STRICTLY use `TIRTL_analyse` conda environment. **DO NOT** install package or dependencies into the `base` environment.
  - **OS Support**: Must support both **Windows** and **macOS**.
  - Must handle missing dependencies gracefully (provide setup instructions if `demultiplex` is missing).
  - Reproducible verification via `scripts/verify.py`.
  - No real data in repo (synthetic only).

## 3) Acceptance Criteria (Testable)

- **AC-001 (Interface)**: `scripts/run_demux.py` exists and is executable. 
  - Accepts `--r1`, `--barcodes`, `--output-dir` arguments.
  - Prints help with `--help`.
- **AC-002 (Functionality)**: Running on synthetic data (e.g. 3 known wells + noise) produces:
  - Precise output files: `well_<row><col>.fastq.gz` for each known well.
  - `unknown.fastq.gz` for noise.
- **AC-003 (Data Integrity)**:
  - Output FASTQ files result in valid gzip files.
  - R1/R2 pairing is preserved (same number of reads in both files per well).
- **AC-004 (Reporting)**:
  - Generates `demux_stats.tsv` with columns: `well_id`, `read_count`, `percent`.
  - Sum of all well counts + unknown count equals total input reads.
- **AC-005 (Automation)**:
  - `scripts/verify.py` runs the full cycle (Synthetic Gen -> Demux -> Check) and exits with code 0.
  - Verification produces a JSON summary log.

## 4) Plan & Task Breakdown

- [x] **T-001: Environment & Mock Setup**
  - Ensure `demultiplex` is installed in `TIRTL_analyse` env (or provide `requirements.txt` for it).
  - Verify tool runs on Windows (powershell/cmd) and macOS (zsh), or document limitations.
  - Update `scripts/generate_synthetic.py` if necessary.

- [x] **T-002: Demux Wrapper Implementation**
  - Create `scripts/run_demux.py`.
  - Implement barcode table parsing (CSV -> `demultiplex` format).
  - Implement the subprocess call to `demultiplex`.
  - Implement file renaming/organizing logic (move from temp to `out/well/`).
  - Implement stats generation.

- [x] **T-003: Verification Logic**
  - Update `scripts/verify.py` to include `verify_demux_well()`.
  - Implement checks for AC-002, AC-003, AC-004.

- [x] **T-004: Documentation & Final Polish**
  - Update `docs/` with usage instructions (emphasizing `conda activate TIRTL_analyse`).
  - Run final full verification.
  - Update Milestone 'Files Changed' and 'Summary'.

## 5) Implementation Notes

- **Dependency**: Takes a dependency on `jfjlaros/demultiplex`. 
- **Environment**: Instructions must remind user to `conda activate TIRTL_analyse`.
- **Naming**: Output files should use `RowCol` format (e.g., `A01`) to match the barcode file `Row`, `Column` fields.
- **Stats**: Simple CSV/TSV is sufficient.

## 6) Verification

- **Command**: `python scripts/verify.py --milestone M-0002`
- **Last Run**: 2026-01-15 11:35
- **Log**: `logs/M-0002-verify-20260115-1135.txt`
- **Result**: 23/23 Checks Passed.

### Scenario:
1. Generate synthetic data (3 wells, 10 reads each, plus random noise).
2. Run `scripts/run_demux.py`.
3. Check presence of 3 named well files + unknown file.
4. Check read counts match exactly (10 per well).
5. Verify no data loss.

## 7) Files Changed

- `scripts/run_demux.py`: [NEW] Wrapper script for well-code demultiplexing (AC-001, AC-002, AC-004).
- `scripts/verify.py`: [MODIFIED] Added `verify_m0002_full_cycle` and supporting checks (AC-005).
- `docs/demux_plan.md`: [MODIFIED] Added usage instructions and environment setup.
- `scripts/patches/tssv-1.1.2-arm64/`: [NEW] ARM64 compatibility patch for tssv dependency.
- `requirements.txt`: [MODIFIED] Added ARM64 installation notes.

## 8) Acceptance Summary

**Status**: ✅ Complete

| AC | Description | Status | Evidence |
|----|-------------|--------|----------|
| AC-001 | Interface (`--r1`, `--barcodes`, `--output-dir`, `--help`) | ✅ Pass | Verification checks 2-6 |
| AC-002 | Functionality (well files + unknown.fastq.gz) | ✅ Pass | Verification checks 7-12 |
| AC-003 | Data Integrity (valid gzip files) | ✅ Pass | Verification checks 13-17 |
| AC-004 | Reporting (demux_stats.tsv with counts) | ✅ Pass | Verification checks 18-23 |
| AC-005 | Automation (verify.py full cycle, JSON log) | ✅ Pass | 23/23 checks passed |

**Final Verification**: 2026-01-15 11:35, 23/23 checks passed  
**Log**: `logs/M-0002-verify-20260115-1135.txt`  
**Summary**: `logs/M-0002-verify-20260115-1135.summary.json`

## 9) Change Log

- **v2.1**: Added strict environment (TIRTL_analyse) and OS (Win/Mac) constraints.
- **v2**: Refined tasks, ACs, and verification limits. Aligned with R3 Pipeline.
- **v1**: Initial Draft.